# Monitoring a construction site with enhanced Sentinel-2 (hyper-ez + hyEnhance)

This notebook enhances Sentinel-2 from 10 m to 2 m with the **P2SR** method of
[hyEnhance](https://github.com/naikp13/hyEnhance) (Naik et al., 2025, *Remote Sensing* 17(11), 1878,
[doi:10.3390/rs17111878](https://doi.org/10.3390/rs17111878)), using **ESRI World Imagery** as the high-resolution guide,
and uses the enhanced images to **track change over a site across several dates**.

| Role in `HSIEnhancer` | hyper-ez (hyperspectral workflow) | This notebook |
|---|---|---|
| low-resolution spectral image (`hsi_path`) | EnMAP 30 m (224 bands) | **Sentinel-2 L2A**, 12 bands on a 10 m grid, one per date |
| high-resolution guide (`msi_path`) | Sentinel-2 10 m, then ESRI 2 m | **ESRI World Imagery** RGB, 2 m (configurable) |

**Inputs**: an AOI polygon file (shapefile `.shp` with its `.shx`, `.dbf` and `.prj`, or a zipped shapefile, GeoPackage or
GeoJSON) and a list of dates.

**Workflow**

1. Load the AOI and build a UTM working grid around it.
2. For each date, find the closest Sentinel-2 L2A scene that covers the AOI and is cloud-free *over the AOI*
   (checked with the scene classification layer, SCL).
3. Read the 12 bands for each date (only the needed window), as surface reflectance on the 10 m grid.
4. Download one ESRI World Imagery guide for the footprint.
5. Enhance every date to 2 m with hyEnhance `HSIEnhancer` and check that each result still matches its 10 m original.
6. Change analysis on the enhanced images: time series of construction-related indices (NDVI, NDBI, BSI), change-vector
   maps between dates, changed area per interval, and a comparison with the same analysis at 10 m.
7. Optional reduced-resolution check of the method (Wald protocol).

> **Important for monitoring.** ESRI World Imagery is a single mosaic with one acquisition date, and the same guide is
> used for every Sentinel-2 date. The 2 m edges therefore come from that one image, while the *spectral* values (and so
> the changes measured here) come from each Sentinel-2 date. A structure that is missing from the ESRI image still shows
> up as a change, but with softer, Sentinel-2-like edges. Read the change maps as "Sentinel-2 change, drawn on a 2 m
> grid", not as independent 2 m observations.

Runs on Google Colab or locally (Python 3.10+). Needs internet access to GitHub, PyPI, `earth-search.aws.element84.com`,
`sentinel-cogs.s3.us-west-2.amazonaws.com` and `services.arcgisonline.com`.

## 1. Setup

In [ ]:
# Clone hyper-ez and hyEnhance next to this notebook (skipped if they already exist)
# and install any missing dependencies.
import importlib.util
import os
import subprocess
import sys

REPOS = {
    "hyper-ez": "https://github.com/naikp13/hyper-ez.git",
    "hyEnhance": "https://github.com/naikp13/hyEnhance.git",  # keep this folder name: hyper-ez imports `hyEnhance.src`
}
REPO_ROOT = os.environ.get("HYPEREZ_REPO_ROOT", os.getcwd())
for name, url in REPOS.items():
    path = os.path.join(REPO_ROOT, name)
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "--depth", "1", url, path], check=True)

for p in (os.path.join(REPO_ROOT, "hyper-ez"), REPO_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

# module name -> pip package
REQUIRED = {
    "numpy": "numpy", "scipy": "scipy", "rasterio": "rasterio", "shapely": "shapely",
    "geopandas": "geopandas", "pyogrio": "pyogrio", "pandas": "pandas",
    "pystac_client": "pystac-client", "requests": "requests", "pytz": "pytz",
    "matplotlib": "matplotlib", "tqdm": "tqdm", "joblib": "joblib", "numba": "numba",
    "pywt": "PyWavelets", "sklearn": "scikit-learn", "skimage": "scikit-image",
}


def has_cv2_contrib():
    # Probe in a separate process so this kernel never caches a cv2 without `ximgproc`
    # (the guided filter lives in opencv-contrib)
    probe = "import cv2, sys; sys.exit(0 if hasattr(cv2, 'ximgproc') else 1)"
    return subprocess.run([sys.executable, "-c", probe], capture_output=True).returncode == 0


missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if not has_cv2_contrib():
    missing.append("opencv-contrib-python")
if missing:
    print("Installing:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if "opencv-contrib-python" in missing and not has_cv2_contrib():
        raise RuntimeError("opencv-contrib-python is installed but cv2 still has no `ximgproc`. Uninstall "
                           "opencv-python / opencv-python-headless, reinstall opencv-contrib-python, then restart the kernel.")
    if "cv2" in sys.modules:
        raise RuntimeError("OpenCV was replaced after it was imported. Restart the kernel and run the notebook again.")
else:
    print("All dependencies are installed.")

In [ ]:
import logging
import math
import warnings
from datetime import datetime, timedelta

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import rasterio
from pystac_client import Client
from rasterio.coords import BoundingBox
from rasterio.crs import CRS
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import from_bounds as window_from_bounds
from shapely.geometry import shape
from skimage.filters import threshold_otsu
from skimage.transform import resize

# hyper-ez: configuration and ESRI World Imagery download
from hyper_ez.config import STAC_API_URL, SENTINEL2_BAND_MAPPING, setup_matplotlib_style
from hyper_ez.image_processing import HyperspectralProcessor

# hyEnhance: the P2SR enhancement method
from hyEnhance.src import HSIEnhancer, HSIDataLoader, PatchProcessor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
setup_matplotlib_style()
plt.rcParams["font.size"] = 12

## 2. Parameters

- `AOI_PATH`: the construction-site polygon. For a shapefile, keep `.shp`, `.shx`, `.dbf` and `.prj` together in one
  folder and point to the `.shp` (a `.zip` of those files works too). Without a `.prj`, coordinates are assumed to be
  WGS84 longitude/latitude. Several polygons are merged into one AOI. On Colab, run the upload cell below first.
- `AOI_BUFFER_M`: margin around the AOI that is also processed. It gives context in the maps and a stable surrounding
  area for the change threshold.
- `DATES`: the dates to monitor. For each one the closest suitable scene within `SEARCH_WINDOW_DAYS` is used.
  Prefer dates from the same season, because crop and grass cycles also change NDVI.
- `MIN_AOI_CLEAR`: the share of AOI pixels that must be free of cloud, cloud shadow and cirrus according to SCL.
- `GUIDE_RADIUS = 4` and `DETAIL_WEIGHT = 0.25` scored best for Sentinel-2 in the reduced-resolution check (section 9);
  hyper-ez uses 10 and 1 for hyperspectral data.
- The ESRI export service returns at most 4096 pixels per side, so the AOI plus buffer can be up to about 8 km across at 2 m.

The defaults monitor the Tesla Gigafactory Berlin-Brandenburg site (Grünheide, Germany), which was built in 2020 and
2021, using the sample AOI written by the next cell.

In [ ]:
AOI_PATH = "aoi/construction_site.shp"
AOI_BUFFER_M = 150

DATES = ["2020-03-25", "2020-06-25", "2020-09-20", "2021-04-20"]
SEARCH_WINDOW_DAYS = 25      # +/- days around each date
MAX_SCENE_CLOUD = 60         # scene-level cloud cover pre-filter, percent
MIN_AOI_CLEAR = 0.95         # required clear fraction inside the AOI

# Grids
S2_RES_M = 10.0
ESRI_RES_M = 2.0

# P2SR / HSIEnhancer parameters
PATCH_SIZE = 12
STRIDE = 2
GUIDE_RADIUS = 4      # hyper-ez default: 10
DETAIL_WEIGHT = 0.25  # hyper-ez default: 1

# Change analysis
NDVI_CHANGE = 0.15        # |dNDVI| above this counts as vegetation loss / gain
CVA_THRESHOLD = None      # change-vector magnitude threshold (reflectance); None = automatic (Otsu, see section 8)

S2_BANDS = list(SENTINEL2_BAND_MAPPING.keys())   # B01..B12 with B8A after B08
OUT_DIR = "s2_monitoring_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SCALE = S2_RES_M / ESRI_RES_M
assert abs(SCALE - round(SCALE)) < 1e-9, "ESRI_RES_M must divide 10 m evenly"
SCALE = int(round(SCALE))
print(f"Scale factor: {SCALE}x  ({S2_RES_M:g} m -> {ESRI_RES_M:g} m), {len(DATES)} dates")

In [ ]:
# Sample AOI (Tesla Gigafactory Berlin-Brandenburg construction site). Skipped when AOI_PATH already exists.
# Replace AOI_PATH above with your own file, or on Colab upload it with the next cell.
from shapely.geometry import Polygon

if not os.path.exists(AOI_PATH):
    os.makedirs(os.path.dirname(AOI_PATH) or ".", exist_ok=True)
    site = Polygon([(13.7855, 52.3905), (13.8065, 52.3905), (13.8095, 52.3985),
                    (13.8030, 52.4035), (13.7855, 52.4020), (13.7855, 52.3905)])
    gpd.GeoDataFrame({"name": ["Gigafactory Berlin-Brandenburg"]}, geometry=[site], crs="EPSG:4326").to_file(AOI_PATH)
    print("Wrote sample AOI:", AOI_PATH)

In [ ]:
# Colab only: upload your own AOI (select the .shp, .shx, .dbf and .prj together, or one .zip).
# Leave UPLOAD_AOI = False to keep AOI_PATH.
UPLOAD_AOI = False

if UPLOAD_AOI:
    from google.colab import files
    os.makedirs("aoi_upload", exist_ok=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join("aoi_upload", name), "wb") as fh:
            fh.write(data)
    main = [n for n in uploaded if n.lower().endswith((".shp", ".zip", ".gpkg", ".geojson", ".json"))]
    assert main, "Upload a .shp (with .shx, .dbf, .prj), a .zip, a .gpkg or a .geojson"
    AOI_PATH = os.path.join("aoi_upload", main[0])
    print("AOI_PATH =", AOI_PATH)

## 3. Load the AOI and build the working grid

In [ ]:
def load_aoi(path):
    gdf = gpd.read_file(path)
    if gdf.empty:
        raise ValueError(f"{path} has no features")
    if gdf.crs is None:
        print("No .prj / CRS found: assuming WGS84 longitude/latitude (EPSG:4326).")
        gdf = gdf.set_crs(4326)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    if not gdf.geom_type.isin(["Polygon", "MultiPolygon"]).all():
        raise ValueError("The AOI must contain polygons")
    return gdf


def snapped_bounds(geom, buffer_m, snap=60.0):
    # Bounds of the buffered AOI, snapped outward to multiples of 60 m, so the 10/20/60 m Sentinel-2
    # pixels and the ESRI pixels all nest exactly
    minx, miny, maxx, maxy = geom.buffer(buffer_m).bounds
    return BoundingBox(math.floor(minx / snap) * snap, math.floor(miny / snap) * snap,
                       math.ceil(maxx / snap) * snap, math.ceil(maxy / snap) * snap)


aoi_gdf = load_aoi(AOI_PATH)
work_crs = CRS.from_user_input(aoi_gdf.estimate_utm_crs())
aoi_utm = aoi_gdf.to_crs(work_crs).geometry.union_all() if hasattr(aoi_gdf.geometry, "union_all") \
    else aoi_gdf.to_crs(work_crs).unary_union
aoi_wgs84 = gpd.GeoSeries([aoi_utm], crs=work_crs).to_crs(4326).iloc[0]
work_bounds = snapped_bounds(aoi_utm, AOI_BUFFER_M)

n_rows = int(round((work_bounds.top - work_bounds.bottom) / S2_RES_M))
n_cols = int(round((work_bounds.right - work_bounds.left) / S2_RES_M))
assert max(n_rows, n_cols) * SCALE <= 4096, "AOI + buffer is too large for one ESRI export: shrink it or raise ESRI_RES_M"


def aoi_mask(res):
    # True inside the AOI on a grid of pixel size `res`
    h = int(round((work_bounds.top - work_bounds.bottom) / res))
    w = int(round((work_bounds.right - work_bounds.left) / res))
    return geometry_mask([aoi_utm], out_shape=(h, w), transform=transform_from_bounds(*work_bounds, w, h), invert=True)


def aoi_outline(ax, res, color="yellow"):
    geoms = aoi_utm.geoms if aoi_utm.geom_type == "MultiPolygon" else [aoi_utm]
    for g in geoms:
        x, y = g.exterior.xy
        ax.plot((np.array(x) - work_bounds.left) / res - 0.5, (work_bounds.top - np.array(y)) / res - 0.5,
                color=color, lw=1.5)


print(f"AOI: {len(aoi_gdf)} feature(s), {aoi_utm.area / 1e4:.1f} ha, working CRS {work_crs}")
print(f"Grid: {n_rows} x {n_cols} px at {S2_RES_M:g} m  ->  {n_rows * SCALE} x {n_cols * SCALE} px at {ESRI_RES_M:g} m")

## 4. Choose one Sentinel-2 scene per date

Candidates come from the AWS Earth Search STAC API: scenes within `SEARCH_WINDOW_DAYS` of the date whose footprint covers
the AOI and whose scene-level cloud cover is below `MAX_SCENE_CLOUD`. They are tried from the closest date outward, and
the first one whose SCL layer marks at least `MIN_AOI_CLEAR` of the AOI as clear is kept. If none passes, the clearest
candidate is kept and flagged.

SCL classes treated as not clear: 0 no data, 1 saturated, 3 cloud shadow, 8 and 9 cloud, 10 cirrus. Very bright new
roofs or concrete are sometimes classed as cloud; look at the true-colour images in section 6 if a date seems to lose
many pixels.

In [ ]:
NOT_CLEAR_SCL = [0, 1, 3, 8, 9, 10]
GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", AWS_NO_SIGN_REQUEST="YES",
                CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif")


def read_asset(href, res, resampling, src_nodata=0):
    # Read only the window of a COG that covers the working grid and resample it onto the grid
    h = int(round((work_bounds.top - work_bounds.bottom) / res))
    w = int(round((work_bounds.right - work_bounds.left) / res))
    out = np.full((h, w), np.nan, dtype=np.float32)
    with rasterio.Env(**GDAL_ENV), rasterio.open(href) as src:
        # the grid's corners in the source CRS (they differ when the scene is in a neighbouring UTM zone)
        l, b, r, t = transform_bounds(work_crs, src.crs, *work_bounds)
        pad = 2 * max(src.res)
        win = window_from_bounds(l - pad, b - pad, r + pad, t + pad, transform=src.transform)
        win = win.round_offsets().round_lengths()
        data = src.read(1, window=win).astype(np.float32)
        data[data == src_nodata] = np.nan
        reproject(data, out, src_transform=src.window_transform(win), src_crs=src.crs, src_nodata=np.nan,
                  dst_transform=transform_from_bounds(*work_bounds, w, h), dst_crs=work_crs,
                  dst_nodata=np.nan, resampling=resampling)
    return out


def aoi_clear_fraction(item):
    scl = read_asset(item.assets["scl"].href, S2_RES_M, Resampling.nearest, src_nodata=-1)
    clear = np.isfinite(scl) & ~np.isin(scl, NOT_CLEAR_SCL)
    return clear[AOI_MASK_10].mean(), clear


def stac_candidates(date):
    t0 = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=pytz.UTC)
    rng = f"{(t0 - timedelta(days=SEARCH_WINDOW_DAYS)):%Y-%m-%d}/{(t0 + timedelta(days=SEARCH_WINDOW_DAYS)):%Y-%m-%d}"
    items = Client.open(STAC_API_URL).search(
        collections=["sentinel-2-l2a"], intersects=aoi_wgs84.__geo_interface__, datetime=rng,
        query={"eo:cloud_cover": {"lt": MAX_SCENE_CLOUD}}).items()
    items = [it for it in items if shape(it.geometry).contains(aoi_wgs84)]
    return sorted(items, key=lambda it: abs(it.datetime - t0))


def select_scene(date):
    best = None
    for item in stac_candidates(date):
        frac, clear = aoi_clear_fraction(item)
        if best is None or frac > best[1]:
            best = (item, frac, clear)
        if frac >= MIN_AOI_CLEAR:
            break
    if best is None:
        raise RuntimeError(f"No Sentinel-2 scene covers the AOI within {SEARCH_WINDOW_DAYS} days of {date}")
    return best


AOI_MASK_10 = aoi_mask(S2_RES_M)
scenes = []
for d in DATES:
    item, frac, clear = select_scene(d)
    flag = "" if frac >= MIN_AOI_CLEAR else "  <- below MIN_AOI_CLEAR, cloudy pixels are masked"
    print(f"{d}: {item.id}  ({item.datetime:%Y-%m-%d}, AOI clear {frac:.0%}){flag}")
    scenes.append(dict(requested=d, item=item, date=item.datetime.date(), clear_frac=frac, clear_10=clear))

## 5. Read Sentinel-2 for every date and download the ESRI guide

Digital numbers are converted to surface reflectance with the scale and offset in the STAC metadata. Scenes from
processing baseline 04.00 onward carry a +1000 offset; it is removed unless Earth Search has already removed it
(`earthsearch:boa_offset_applied`). The 20 m and 60 m bands are resampled bilinearly to 10 m.

The ESRI guide is downloaded once with hyper-ez's `HyperspectralProcessor.acquire_high_resolution_imagery`.

In [ ]:
def band_scale_offset(asset, item):
    # Earth Search flags scenes whose BOA offset it already removed; applying -0.1 again would make dark pixels negative
    rb = (asset.extra_fields.get("raster:bands") or [{}])[0]
    scale = rb.get("scale", 1e-4)
    if item.properties.get("earthsearch:boa_offset_applied", False):
        return scale, 0.0
    if "offset" in rb:
        return scale, rb["offset"]
    baseline = float(item.properties.get("s2:processing_baseline", "0") or 0)
    return scale, (-0.1 if baseline >= 4.0 else 0.0)


def read_s2_reflectance(item):
    layers = []
    for b in S2_BANDS:
        asset = item.assets[SENTINEL2_BAND_MAPPING[b]]
        scale, offset = band_scale_offset(asset, item)
        layers.append(read_asset(asset.href, S2_RES_M, Resampling.bilinear) * scale + offset)
    return np.stack(layers, axis=-1).astype(np.float32)


def write_geotiff(path, data_hwc, band_names=None, dtype="float32"):
    h, w, c = data_hwc.shape
    profile = dict(driver="GTiff", height=h, width=w, count=c, dtype=dtype, crs=work_crs,
                   transform=transform_from_bounds(*work_bounds, w, h), compress="deflate",
                   tiled=True, blockxsize=256, blockysize=256)
    if dtype == "float32":
        profile["nodata"] = np.nan
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(np.moveaxis(data_hwc, -1, 0).astype(dtype))
        for i, name in enumerate(band_names or [], start=1):
            dst.set_band_description(i, name)


for sc in scenes:
    sc["s2"] = read_s2_reflectance(sc["item"])
    sc["s2_path"] = os.path.join(OUT_DIR, f"sentinel2_10m_{sc['date']}.tif")
    write_geotiff(sc["s2_path"], sc["s2"], S2_BANDS)
    print(f"{sc['date']}: {sc['s2'].shape} -> {sc['s2_path']}")

# ESRI World Imagery guide (one image for all dates)
downloaded = HyperspectralProcessor().acquire_high_resolution_imagery(work_bounds, work_crs, target_resolution=ESRI_RES_M)
with rasterio.open(downloaded) as src:
    esri = np.moveaxis(src.read(), 0, -1)[..., :3]  # keep R, G, B
os.remove(downloaded)
esri_path = os.path.join(OUT_DIR, f"esri_{ESRI_RES_M:g}m.tif")
write_geotiff(esri_path, esri, ["Red", "Green", "Blue"], dtype="uint8")
assert esri.shape[:2] == (n_rows * SCALE, n_cols * SCALE), f"Grid mismatch: ESRI {esri.shape[:2]}"
print(f"ESRI guide: {esri.shape} -> {esri_path}")

## 6. Enhance every date to 2 m

`HSIEnhancer(guide, sentinel2)` runs the three P2SR stages: the patch-wise sparse residual (`PatchProcessor`), injection
of the guide's high-frequency detail with each band's mean and standard deviation restored (`HSIUpsampler`), and a
per-band guided filter with the ESRI image as guide. Each date takes about a minute for a 300 × 300 grid.

After each run the 2 m result is averaged back to 10 m and compared with its original (SAM in degrees, ERGAS; lower is
better for both, and ERGAS under about 3 is usually considered good). This confirms the enhancement kept each date's
spectra, which the change analysis relies on.

In [ ]:
def block_mean(cube, f):
    h, w, c = cube.shape
    return cube[: h // f * f, : w // f * f].reshape(h // f, f, w // f, f, c).mean(axis=(1, 3))


def quality_metrics(ref, est, ratio):
    valid = np.isfinite(ref).all(-1) & np.isfinite(est).all(-1)
    r, e = ref[valid].astype(np.float64), est[valid].astype(np.float64)
    rmse_b = np.sqrt(((r - e) ** 2).mean(0))
    cc_b = np.array([np.corrcoef(r[:, i], e[:, i])[0, 1] for i in range(r.shape[1])])
    cos = (r * e).sum(1) / (np.linalg.norm(r, axis=1) * np.linalg.norm(e, axis=1) + 1e-12)
    return dict(RMSE=np.sqrt((rmse_b ** 2).mean()), PSNR=20 * np.log10(r.max() / (np.sqrt((rmse_b ** 2).mean()) + 1e-12)),
                SAM=np.degrees(np.arccos(np.clip(cos, -1, 1))).mean(),
                ERGAS=100.0 / ratio * np.sqrt(np.mean((rmse_b / (np.abs(r.mean(0)) + 1e-12)) ** 2)), CC=cc_b.mean())


def enhance(guide_path, s2_path):
    enhancer = HSIEnhancer(guide_path, s2_path)
    kept = np.isnan(HSIDataLoader.load_image(s2_path)).mean(axis=(0, 1)) <= 0.5
    if not kept.all():
        raise RuntimeError(f"{s2_path}: bands {np.array(S2_BANDS)[~kept].tolist()} are mostly no-data")
    out = enhancer.fuse_to_enhance(patch_size=PATCH_SIZE, stride=STRIDE,
                                   guide_radius=GUIDE_RADIUS, detail_weight=DETAIL_WEIGHT)
    return out.astype(np.float32), enhancer


for sc in scenes:
    sc["enh"], enhancer = enhance(esri_path, sc["s2_path"])
    sc["enh_path"] = os.path.join(OUT_DIR, f"sentinel2_enhanced_{ESRI_RES_M:g}m_{sc['date']}.tif")
    write_geotiff(sc["enh_path"], sc["enh"], S2_BANDS)
    sc["consistency"] = quality_metrics(sc["s2"], block_mean(sc["enh"], SCALE), SCALE)
    q = sc["consistency"]
    print(f"{sc['date']}: {sc['enh'].shape} -> {sc['enh_path']}   SAM {q['SAM']:.2f} deg, ERGAS {q['ERGAS']:.2f}")

**Diagnostic: is the patch-wise residual step contributing?** `PatchProcessor.process_patch` catches any error, logs it
and skips the patch, so a failing step does not stop the run. This cell runs a few patches of the last date in the
notebook process and prints the first error. If every patch fails, the result comes from the detail injection and the
guided filter only.

In [ ]:
class _Collect(logging.Handler):
    def __init__(self):
        super().__init__(); self.messages = []
    def emit(self, record):
        self.messages.append(record.getMessage())

probe = PatchProcessor(enhancer.hsi, enhancer.msi, enhancer.n_components, enhancer.n_atoms, enhancer.lambda_reg)
probe.f = SCALE
probe.msi_lr = resize(enhancer.msi, (*enhancer.hsi.shape[:2], enhancer.msi.shape[2]), order=2, preserve_range=True)
collector = _Collect(); logging.getLogger().addHandler(collector)
step = max(1, (min(enhancer.hsi.shape[:2]) - PATCH_SIZE) // 4)
coords = [(x, x) for x in range(0, min(enhancer.hsi.shape[:2]) - PATCH_SIZE + 1, step)][:5]
ok = sum(probe.process_patch(x, y, PATCH_SIZE) is not None for x, y in coords)
logging.getLogger().removeHandler(collector)
print(f"{ok} of {len(coords)} probe patches produced a sparse residual.")
if ok < len(coords) and collector.messages:
    print("First error:", collector.messages[0])

### Images over time

In [ ]:
def stretch(rgb, lo_hi):
    lo, hi = lo_hi
    return np.clip((rgb - lo) / (hi - lo + 1e-12), 0, 1)


def bands(cube, names):
    return cube[..., [S2_BANDS.index(n) for n in names]]


TRUE_COLOR = ["B04", "B03", "B02"]
# One stretch for all dates (from the first date), so brightness differences between dates are real
ref_rgb = bands(scenes[0]["s2"], TRUE_COLOR)
LIMITS = (np.nanpercentile(ref_rgb, 2, axis=(0, 1)), np.nanpercentile(ref_rgb, 98, axis=(0, 1)))

n = len(scenes)
fig, axes = plt.subplots(2, n + 1, figsize=(4.6 * (n + 1), 9.4))
for j, sc in enumerate(scenes):
    lr = np.repeat(np.repeat(bands(sc["s2"], TRUE_COLOR), SCALE, 0), SCALE, 1)
    axes[0, j].imshow(stretch(lr, LIMITS), interpolation="nearest")
    axes[1, j].imshow(stretch(bands(sc["enh"], TRUE_COLOR), LIMITS), interpolation="nearest")
    axes[0, j].set_title(f"{sc['date']}\nSentinel-2 {S2_RES_M:g} m")
    axes[1, j].set_title(f"Enhanced {ESRI_RES_M:g} m")
axes[0, n].imshow(esri); axes[0, n].set_title(f"ESRI guide {ESRI_RES_M:g} m\n(single date)")
axes[1, n].axis("off")
for ax in axes.ravel():
    if ax.has_data():
        aoi_outline(ax, ESRI_RES_M)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 7. Construction indices over time

| Index | Formula (Sentinel-2 bands) | What it shows on a construction site |
|---|---|---|
| NDVI | (B08 − B04) / (B08 + B04) | vegetation; drops when the site is cleared |
| BSI (bare soil index) | ((B11 + B04) − (B08 + B02)) / ((B11 + B04) + (B08 + B02)) | exposed soil and earthworks; rises after clearing |
| NDBI (built-up index) | (B11 − B08) / (B11 + B08) | built surfaces such as concrete and roofs; rises as structures appear |

B11 is a 20 m band, so BSI and NDBI carry less true spatial detail than NDVI. Statistics use only clear pixels inside the AOI.

In [ ]:
def indices(cube):
    b = {n: cube[..., S2_BANDS.index(n)] for n in ("B02", "B04", "B08", "B11")}
    eps = 1e-6
    return {
        "NDVI": (b["B08"] - b["B04"]) / (b["B08"] + b["B04"] + eps),
        "BSI": ((b["B11"] + b["B04"]) - (b["B08"] + b["B02"])) / ((b["B11"] + b["B04"]) + (b["B08"] + b["B02"]) + eps),
        "NDBI": (b["B11"] - b["B08"]) / (b["B11"] + b["B08"] + eps),
    }


AOI_MASK_HR = aoi_mask(ESRI_RES_M)
for sc in scenes:
    sc["clear_hr"] = np.repeat(np.repeat(sc["clear_10"], SCALE, 0), SCALE, 1)
    sc["idx"] = indices(sc["enh"])
    sc["idx_10"] = indices(sc["s2"])
    sel = AOI_MASK_HR & sc["clear_hr"]
    sc["idx_median"] = {k: float(np.nanmedian(v[sel])) for k, v in sc["idx"].items()}
    # share of the clear AOI that is not vegetated (NDVI < 0.3): cleared, bare or built
    sc["nonveg_pct"] = 100 * float((sc["idx"]["NDVI"][sel] < 0.3).mean())

dates = [pd.Timestamp(sc["date"]) for sc in scenes]
INDEX_COLORS = {"NDVI": "#2a78d6", "BSI": "#eb6834", "NDBI": "#1baf7a"}
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(17, 4.8))
for k, col in INDEX_COLORS.items():
    vals = [sc["idx_median"][k] for sc in scenes]
    ax.plot(dates, vals, "-o", color=col, lw=2, ms=8, label=k)
    ax.annotate(k, (dates[-1], vals[-1]), xytext=(8, 0), textcoords="offset points", va="center", color="#52514e")
ax.axhline(0, color="#999999", lw=1)
ax.set_ylabel("Median over AOI")
ax.set_title("Construction indices (enhanced, clear AOI pixels)")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.08, 1))
ax2.plot(dates, [sc["nonveg_pct"] for sc in scenes], "-o", color="#2a78d6", lw=2, ms=8)
for d_, sc in zip(dates, scenes):
    ax2.annotate(f"{sc['nonveg_pct']:.0f}%", (d_, sc["nonveg_pct"]), xytext=(0, 8), textcoords="offset points", ha="center")
ax2.set_ylim(0, 105); ax2.set_ylabel("% of clear AOI")
ax2.set_title("Non-vegetated share of the AOI (NDVI < 0.3)")
for a in (ax, ax2):
    a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m")); a.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, n, figsize=(4.6 * n, 13), squeeze=False)
ranges = {"NDVI": (-0.2, 0.9), "BSI": (-0.4, 0.4), "NDBI": (-0.5, 0.4)}
cmaps = {"NDVI": "Greens", "BSI": "Oranges", "NDBI": "Purples"}
for i, k in enumerate(("NDVI", "BSI", "NDBI")):
    for j, sc in enumerate(scenes):
        im = axes[i, j].imshow(sc["idx"][k], cmap=cmaps[k], vmin=ranges[k][0], vmax=ranges[k][1])
        aoi_outline(axes[i, j], ESRI_RES_M, color="black")
        axes[i, j].set_title(f"{k}  {sc['date']}"); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.8, pad=0.01)
plt.show()

## 8. Change between dates

For each interval (each date to the next, plus first to last):

- **Change-vector magnitude (CVA)**: the length of the 12-band reflectance difference per pixel. It catches any kind
  of change: clearing, earthworks, new roofs, paving.
- **Change mask**: CVA above a threshold. With `CVA_THRESHOLD = None` the threshold is set automatically by Otsu's
  method on the CVA values of the whole working area (AOI plus buffer), with a floor of 0.05 so that two near-identical
  dates do not produce spurious change.
- **Change type** inside the mask, checked in this order:
  - *vegetation cleared*: NDVI drops by more than `NDVI_CHANGE` from a vegetated start (NDVI > 0.3);
  - *new bright surface*: visible brightness (mean of B02, B03, B04) rises by more than 0.04 and the end state is
    not vegetated (NDVI < 0.2): new roofs and concrete, but also freshly graded light-coloured ground;
  - *earthworks / bare soil*: BSI rises by more than 0.05 and the end state is not vegetated (NDVI < 0.3);
  - *vegetation regrowth*: NDVI rises by more than `NDVI_CHANGE` to a vegetated end state (NDVI > 0.3);
  - *other change*: anything else above the threshold (for example seasonal change in the surroundings, or material
    changing on an already built surface).

Pixels that are cloudy on either date are excluded. The same analysis at 10 m is shown next to the 2 m one.

In [ ]:
CHANGE_CLASSES = ["no change", "vegetation cleared", "new bright surface", "earthworks / bare soil",
                  "vegetation regrowth", "other change"]
CHANGE_COLORS = ["#f0efec", "#e34948", "#6250d6", "#eda100", "#008300", "#a9a8a0"]
PIXEL_HA = {ESRI_RES_M: ESRI_RES_M ** 2 / 1e4, S2_RES_M: S2_RES_M ** 2 / 1e4}


def change_between(a, b, res, threshold=CVA_THRESHOLD):
    hr = res == ESRI_RES_M
    ca, cb = (a["enh"], b["enh"]) if hr else (a["s2"], b["s2"])
    ia, ib = (a["idx"], b["idx"]) if hr else (a["idx_10"], b["idx_10"])
    clear = (a["clear_hr"] & b["clear_hr"]) if hr else (a["clear_10"] & b["clear_10"])
    inside = (AOI_MASK_HR if hr else AOI_MASK_10)

    cva = np.linalg.norm(cb - ca, axis=-1)
    cva[~clear] = np.nan
    if threshold is None:
        vals = cva[np.isfinite(cva)]
        threshold = max(float(threshold_otsu(vals)), 0.05)
    changed = np.isfinite(cva) & (cva > threshold)

    d_ndvi, d_bsi = ib["NDVI"] - ia["NDVI"], ib["BSI"] - ia["BSI"]
    d_bright = cb[..., 1:4].mean(-1) - ca[..., 1:4].mean(-1)   # B02, B03, B04
    rules = [  # (class index, condition), first match wins
        (1, (d_ndvi < -NDVI_CHANGE) & (ia["NDVI"] > 0.3)),
        (2, (d_bright > 0.04) & (ib["NDVI"] < 0.2)),
        (3, (d_bsi > 0.05) & (ib["NDVI"] < 0.3)),
        (4, (d_ndvi > NDVI_CHANGE) & (ib["NDVI"] > 0.3)),
        (5, np.ones(cva.shape, bool)),
    ]
    cls = np.zeros(cva.shape, np.uint8)
    todo = changed.copy()
    for c, cond in rules:
        hit = todo & cond
        cls[hit] = c
        todo &= ~hit
    cls[~clear] = 255

    valid = inside & clear
    area = {CHANGE_CLASSES[c]: float(((cls == c) & valid).sum() * PIXEL_HA[res]) for c in range(1, len(CHANGE_CLASSES))}
    return dict(cva=cva, cls=cls, threshold=threshold, area_ha=area,
                changed_ha=sum(area.values()), changed_pct=100 * float((changed & valid).sum()) / max(valid.sum(), 1),
                aoi_valid_ha=float(valid.sum() * PIXEL_HA[res]))


pairs = [(i, i + 1) for i in range(n - 1)]
if n > 2:
    pairs.append((0, n - 1))
changes = []
for i, j in pairs:
    hr = change_between(scenes[i], scenes[j], ESRI_RES_M)
    lr = change_between(scenes[i], scenes[j], S2_RES_M, threshold=hr["threshold"])  # same threshold at both scales
    label = f"{scenes[i]['date']} -> {scenes[j]['date']}"
    changes.append(dict(i=i, j=j, label=label, hr=hr, lr=lr))
    write_geotiff(os.path.join(OUT_DIR, f"change_cva_{scenes[i]['date']}_{scenes[j]['date']}.tif"), hr["cva"][..., None], ["CVA"])
    write_geotiff(os.path.join(OUT_DIR, f"change_class_{scenes[i]['date']}_{scenes[j]['date']}.tif"),
                  hr["cls"][..., None], ["class"], dtype="uint8")
    print(f"{label}: threshold {hr['threshold']:.3f}, changed {hr['changed_ha']:.1f} ha "
          f"({hr['changed_pct']:.0f}% of clear AOI) at {ESRI_RES_M:g} m; {lr['changed_ha']:.1f} ha at {S2_RES_M:g} m")

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

cls_cmap = ListedColormap(CHANGE_COLORS)


def show_classes(ax, cls):
    img = np.ma.masked_equal(cls, 255)
    ax.imshow(img, cmap=cls_cmap, vmin=0, vmax=len(CHANGE_CLASSES) - 1, interpolation="nearest")


for ch in changes:
    a, b = scenes[ch["i"]], scenes[ch["j"]]
    fig, axes = plt.subplots(1, 4, figsize=(21, 5.6))
    axes[0].imshow(stretch(bands(a["enh"], TRUE_COLOR), LIMITS)); axes[0].set_title(f"{a['date']} enhanced")
    axes[1].imshow(stretch(bands(b["enh"], TRUE_COLOR), LIMITS)); axes[1].set_title(f"{b['date']} enhanced")
    show_classes(axes[2], np.repeat(np.repeat(ch["lr"]["cls"], SCALE, 0), SCALE, 1))
    axes[2].set_title(f"Change at {S2_RES_M:g} m")
    show_classes(axes[3], ch["hr"]["cls"]); axes[3].set_title(f"Change at {ESRI_RES_M:g} m (enhanced)")
    for ax in axes:
        aoi_outline(ax, ESRI_RES_M, color="black" if ax in axes[2:] else "yellow")
        ax.set_xticks([]); ax.set_yticks([])
    fig.legend(handles=[Patch(color=c, label=l) for c, l in zip(CHANGE_COLORS, CHANGE_CLASSES)] +
               [Patch(facecolor="white", edgecolor="#999999", label="cloud / no data")],
               loc="lower center", ncol=7, frameon=False)
    fig.suptitle(ch["label"], fontweight="bold")
    plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

### Changed area per interval and summary table

In [ ]:
consecutive = [ch for ch in changes if ch["j"] == ch["i"] + 1]
x = np.arange(len(consecutive))
fig, ax = plt.subplots(figsize=(max(7, 3 * len(consecutive)), 4.8))
bottom = np.zeros(len(consecutive))
for c in range(1, len(CHANGE_CLASSES)):
    vals = np.array([ch["hr"]["area_ha"][CHANGE_CLASSES[c]] for ch in consecutive])
    ax.bar(x, vals, bottom=bottom, width=0.55, color=CHANGE_COLORS[c], label=CHANGE_CLASSES[c],
           edgecolor="white", linewidth=2)
    bottom += vals
for xi, total in zip(x, bottom):
    ax.annotate(f"{total:.1f} ha", (xi, total), xytext=(0, 4), textcoords="offset points", ha="center")
ax.set_xticks(x); ax.set_xticklabels([ch["label"].replace(" -> ", "\nto ") for ch in consecutive])
ax.set_ylim(0, bottom.max() * 1.15 + 1e-6)
ax.set_ylabel("Changed area inside AOI (ha)")
ax.set_title(f"Change per interval (enhanced {ESRI_RES_M:g} m)")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1)); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

rows = []
for k, sc in enumerate(scenes):
    prev = next((ch for ch in consecutive if ch["j"] == k), None)
    rows.append({
        "requested_date": sc["requested"], "scene_date": str(sc["date"]), "scene_id": sc["item"].id,
        "aoi_clear_pct": round(100 * sc["clear_frac"], 1),
        **{f"{name}_median": round(v, 3) for name, v in sc["idx_median"].items()},
        "non_vegetated_pct": round(sc["nonveg_pct"], 1),
        "SAM_vs_10m_deg": round(sc["consistency"]["SAM"], 2), "ERGAS_vs_10m": round(sc["consistency"]["ERGAS"], 2),
        "changed_since_prev_ha": round(prev["hr"]["changed_ha"], 2) if prev else None,
        "changed_since_prev_ha_10m": round(prev["lr"]["changed_ha"], 2) if prev else None,
    })
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT_DIR, "monitoring_summary.csv"), index=False)
if n > 2:
    total = changes[-1]["hr"]
    print(f"First to last date ({changes[-1]['label']}): {total['changed_ha']:.1f} ha changed "
          f"({total['changed_pct']:.0f}% of the clear AOI): " +
          ", ".join(f"{k} {v:.1f} ha" for k, v in total["area_ha"].items()))
summary

## 9. Optional: reduced-resolution check of the method (Wald protocol)

There is no real 2 m Sentinel-2 image to compare with, so this tests the method where the truth is known: the first
date is degraded to 50 m, enhanced back to 10 m with its own 10 m RGB as the guide, and compared with the real 10 m bands
next to plain bicubic upsampling. Set `RUN_WALD_CHECK = True` to run it.

In [ ]:
RUN_WALD_CHECK = False

if RUN_WALD_CHECK:
    f = SCALE
    s2_0 = scenes[0]["s2"]
    h, w = (s2_0.shape[0] // f) * f, (s2_0.shape[1] // f) * f
    truth = s2_0[:h, :w]
    low = block_mean(truth, f)
    rgb = bands(truth, TRUE_COLOR)
    lo, hi = np.nanpercentile(rgb, 2, axis=(0, 1)), np.nanpercentile(rgb, 98, axis=(0, 1))
    guide = (np.clip((rgb - lo) / (hi - lo), 0, 1) * 254 + 1).astype(np.uint8)
    extent = (work_bounds.left, work_bounds.top - h * S2_RES_M, work_bounds.left + w * S2_RES_M, work_bounds.top)
    for path, arr, dt in (("wald_s2_low.tif", low, "float32"), ("wald_guide_rgb.tif", guide, "uint8")):
        tr = transform_from_bounds(*extent, arr.shape[1], arr.shape[0])
        with rasterio.open(os.path.join(OUT_DIR, path), "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                           count=arr.shape[2], dtype=dt, crs=work_crs, transform=tr) as dst:
            dst.write(np.moveaxis(arr, -1, 0).astype(dt))
    wald_enh, _ = enhance(os.path.join(OUT_DIR, "wald_guide_rgb.tif"), os.path.join(OUT_DIR, "wald_s2_low.tif"))
    bicubic = resize(low, truth.shape, order=3, mode="edge", preserve_range=True).astype(np.float32)
    print(f"{'Method':<18}{'RMSE':>9}{'PSNR dB':>9}{'SAM deg':>9}{'ERGAS':>8}{'CC':>8}")
    for name, est in (("Bicubic", bicubic), ("P2SR (hyEnhance)", wald_enh)):
        r = quality_metrics(truth, est, f)
        print(f"{name:<18}{r['RMSE']:>9.4f}{r['PSNR']:>9.2f}{r['SAM']:>9.2f}{r['ERGAS']:>8.2f}{r['CC']:>8.3f}")

## 10. Outputs and notes

Files in `s2_monitoring_outputs/`:

| File | Content |
|---|---|
| `sentinel2_10m_<date>.tif` | 12-band Sentinel-2 L2A reflectance on the 10 m grid, per date |
| `sentinel2_enhanced_2m_<date>.tif` | enhanced 12-band reflectance at 2 m, per date (band names in the band descriptions) |
| `esri_2m.tif` | ESRI World Imagery guide |
| `change_cva_<date1>_<date2>.tif` | change-vector magnitude at 2 m |
| `change_class_<date1>_<date2>.tif` | change classes at 2 m: 0 none, 1 vegetation cleared, 2 new bright surface, 3 earthworks / bare soil, 4 vegetation regrowth, 5 other, 255 cloud |
| `monitoring_summary.csv` | per date: scene, AOI clear %, index medians, consistency with 10 m, changed area since the previous date |

**Tuning and interpretation**

- Raise `CVA_THRESHOLD` (for example 0.08) if stable areas show up as change, or lower it to catch subtle change.
- Compare dates from the same season where you can; phenology alone moves NDVI and can be counted as vegetation change.
- `DETAIL_WEIGHT` sets how much ESRI texture is injected. Keep it low for monitoring so the static guide does not
  dominate; the change maps are driven by Sentinel-2 spectra either way, because the same guide is used for all dates.
- For a guide closer in time to each date, ESRI's World Imagery Wayback archive holds dated past versions of the
  basemap; swapping it in per date would sharpen new structures, at the cost of a different download routine.
- Check the ESRI imagery terms of use before redistributing derived products.

**Citation**: Naik, P., Chakraborty, R., Thiele, S., & Gloaguen, R. (2025). Scalable Hyperspectral Enhancement via
Patch-Wise Sparse Residual Learning: Insights from Super-Resolved EnMAP Data. *Remote Sensing*, 17(11), 1878.
https://doi.org/10.3390/rs17111878